# Visa Credit Card Agreement: Synthetic QA Generation

This notebook adapts notebook 02 to the Visa credit card agreement. It uses the complete extracted document and **does not create a train/test split**. Every paragraph is marked `all` and is eligible for generation.

The notebook writes raw strategy samples, instruction back-translation samples, and one combined overall QA corpus.

In [20]:
import importlib.util
import json
import logging
import os
from pathlib import Path
from aieng.syn_data.text import (
    LLMSettings,
    OpenAICompatibleClient,
    load_implementation_dotenv,
    use_repo_root,
)

from aieng.syn_data.text import (
    effective_synthetic_target,
    generate_grounded_training_corpus,
    generate_raw_synthetic_corpus,
    load_implementation_dotenv,
    save_typed_jsonl,
    use_repo_root,
)
from openai import OpenAI
from rich.console import Console
from rich.table import Table


use_repo_root(Path.cwd())
load_implementation_dotenv()
script_path = Path("implementations/qa_text_generation/scripts/generate_visa_qa.py").resolve()
module_spec = importlib.util.spec_from_file_location("generate_visa_qa", script_path)
if module_spec is None or module_spec.loader is None:
    raise ImportError(f"Could not load Visa QA helper from {script_path}")
visa_qa = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(visa_qa)
DEFAULT_MIN_CHARS = visa_qa.DEFAULT_MIN_CHARS
DEFAULT_MIN_OVERLAP = visa_qa.DEFAULT_MIN_OVERLAP
DEFAULT_QUESTIONS_PER_PARAGRAPH = visa_qa.DEFAULT_QUESTIONS_PER_PARAGRAPH
build_unsplit_paragraphs = visa_qa.build_unsplit_paragraphs
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
console = Console(width=110)


class NotebookOpenAIClient:
    """Adapter for the direct OpenAI SDK used by the PDF QA workflow."""

    def __init__(self, model="gpt-4o"):
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY must be set before generation.")
        self.model = os.getenv("TEACHER_MODEL", model)
        self.client = OpenAI(
            api_key=api_key,
            base_url=os.getenv("VISA_OPENAI_BASE_URL", "https://api.openai.com/v1"),
        )

    def complete_json(self, prompt, *, system=None, temperature=0.0, max_tokens=512):
        """Request a JSON object from the direct OpenAI client."""
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content
        if not content:
            raise ValueError(f"OpenAI model {self.model} returned empty content.")
        payload = json.loads(content)
        if not isinstance(payload, dict):
            raise ValueError(f"Expected a JSON object, got {type(payload).__name__}.")
        return payload

In [16]:
PDF_PATH = Path("implementations/qa_text_generation/data/documents/Visa_Credit_Card_Agreement.pdf")
OUTPUT_DIR = Path("implementations/qa_text_generation/data/visa_synthetic")
QUESTIONS_PER_PARAGRAPH = DEFAULT_QUESTIONS_PER_PARAGRAPH
TARGET_SIZE = None  # None means one instruction-back-translation sample per paragraph.

paragraphs = build_unsplit_paragraphs(PDF_PATH, min_chars=DEFAULT_MIN_CHARS)
assert paragraphs, "The Visa agreement produced no paragraphs."
assert all(paragraph.split.value == "all" for paragraph in paragraphs)

table = Table(title="Visa Agreement Paragraphs", show_lines=True)
table.add_column("Field", style="bold cyan")
table.add_column("Value")
table.add_row("Source", str(PDF_PATH))
table.add_row("Paragraphs", str(len(paragraphs)))
table.add_row("Split", "all (no train/test split)")
table.add_row("Minimum chunk size", str(DEFAULT_MIN_CHARS))
console.print(table)

                                        Visa Agreement Paragraphs                                        
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field              ┃ Value                                                                            ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Source             │ implementations/qa_text_generation/data/documents/Visa_Credit_Card_Agreement.pdf │
├────────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ Paragraphs         │ 8                                                                                │
├────────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ Split              │ all (no train/test split)                                                        │
├────────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ Minimum chunk size │ 80                                                                               │
└────────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

## Inspect the extracted chunks

The chunks retain stable paragraph IDs and the explicit `all` split label. Review these chunks before spending teacher-model tokens.

In [21]:
for paragraph in paragraphs[:3]:
    console.print(f"\n[bold cyan]{paragraph.para_id}[/bold cyan] ({paragraph.split.value})")
    console.print(paragraph.text[:1000] + (" ..." if len(paragraph.text) > 1000 else ""))

visa_credit_card_agreement::p0000 (all)

©CUNA Mutual Group 1991, 2006, 09, 10, 12 All Rights Reserved 
DGAEL6 (MXC501 CCM002)-e

visa_credit_card_agreement::p0001 (all)

VISA
CONSUMER CREDIT CARD AGREEMENT

visa_credit_card_agreement::p0002 (all)

In this Agreement, “Agreement” means this Consumer Credit Card Agreement. “Disclosure” means the Credit Card 
Account Opening Disclosure. The Account Opening Disclosure is incorporated into this Consumer Credit Card 
Agreement 
and is part of the Agreement. In this Agreement the words "you," “your,” and "yours" mean each and all of 
those who 
agree to be bound by this Agreement; "card" means the Visa credit card and any duplicates, renewals, or 
substitutions the 
Credit Union issues to you; "account" means your Visa credit card line-of-credit account with the Credit 
Union; and "Credit 
Union" means the Credit Union whose name appears on this Agreement or anyone to whom the Credit Union 
transfers 
this Agreement. 
1. USING YOUR ACCOUNT — If you are approved for an account, the Credit Union will establish a line of credit 
for you. 
You agree that your credit limit is the maximum amount (purchases, cash advances, finance charges, plus "other
charges") which you will have outstanding on yo ...

In [23]:
def make_client(model_var: str, default_model: str) -> OpenAICompatibleClient:
    return OpenAICompatibleClient(LLMSettings.from_env(model_var=model_var, default_model=default_model))


generator_client = make_client("RAG_GENERATOR_MODEL", "gpt-4o")
answer_client = make_client("RAG_ANSWER_MODEL", "gemini-3.5-flash")
judge_client = make_client("RAG_JUDGE_MODEL", "gpt-4o")
for role, client in ("generator", generator_client), ("answerer", answer_client), ("judge", judge_client):
    console.print(f"{role:10s} {client.settings.model}")


generator  gpt-4o

answerer   gemini-3.5-flash

judge      gpt-4o

In [26]:
ibt_target = TARGET_SIZE if TARGET_SIZE is not None else effective_synthetic_target(
    paragraphs,
    one_per_paragraph=True,
)
back_translation_samples = generate_grounded_training_corpus(
    generator_client,
    paragraphs,
    target_size=ibt_target,
    min_overlap=DEFAULT_MIN_OVERLAP,
)
console.print(f"Back-translation target: {ibt_target}")
console.print(f"Back-translation samples: {len(back_translation_samples)}")

Back-translation target: 8

Back-translation samples: 8

In [27]:
all_samples = raw_samples + back_translation_samples
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_typed_jsonl(OUTPUT_DIR / "visa_qa_paragraphs.jsonl", paragraphs, to_dict=lambda item: item.to_dict())
save_typed_jsonl(OUTPUT_DIR / "visa_qa_raw.jsonl", raw_samples, to_dict=lambda item: item.to_dict())
save_typed_jsonl(
    OUTPUT_DIR / "visa_qa_back_translation.jsonl",
    back_translation_samples,
    to_dict=lambda item: item.to_dict(),
)
save_typed_jsonl(OUTPUT_DIR / "visa_qa_all.jsonl", all_samples, to_dict=lambda item: item.to_dict())

summary = {
    "source": str(PDF_PATH),
    "paragraphs": len(paragraphs),
    "split": "all",
    "raw_count": len(raw_samples),
    "back_translation_count": len(back_translation_samples),
    "overall_count": len(all_samples),
}
(OUTPUT_DIR / "visa_qa_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
console.print_json(json.dumps(summary))

NameError: name 'raw_samples' is not defined

## Review generated examples

The combined file is the overall unsplit corpus. Review samples for grounding and correctness before using them for fine-tuning.

In [ ]:
for sample in all_samples[:5]:
    console.print(f"\n[bold cyan]Q:[/bold cyan] {sample.question}")
    console.print(f"[bold magenta]A:[/bold magenta] {sample.gold_answer[:1000]}")
    console.print(f"[dim]Strategy: {sample.metadata.get('generation_strategy')} | Split: {sample.split.value}[/dim]")